# Build a transient spectral training dataset

Run this notebook from top to bottom. It contains all the selection and export code; **no companion Python script is needed**.

1. Set the paths below to your four CSVs and output folder.
2. Run the selection and review the labels and mapping gaps.
3. Supply the exact classified-spectrum paths, then enable extraction in a DESI Python kernel.

The supplied tables yield **726 selected observation records**, before matching to actual spectra. The notebook does not perform host subtraction. Uncertain labels and conflicting observations are saved for review. No actual spectra have been exported in the saved local run.

## 1. Paths and environment

For NERSC, upload this notebook and the four CSVs together, or set `SOURCE_DIR` to their directory. Use a kernel with **pandas and numpy**; the export also needs **desispec** and access to the source FITS files. No packages are installed by the notebook.

The local path fallback below is only for this workspace. All output paths can be changed. Existing matching selection results are reused; a changed input requires a new output folder. Spectrum output must be a new folder.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import io
import contextlib
import re
import uuid
from urllib.parse import urlparse, parse_qs, quote
from collections import Counter, defaultdict
import pandas as pd
from IPython.display import display

GOOGLE_SHEETS = {
    'prelim_final_table.csv': 'https://docs.google.com/spreadsheets/d/1yBusws08QtusggcULzrbFxNN99nLHKb6nwGmpWBqQdA/edit?gid=974447016#gid=974447016',
    'dts_classifications_snr5.csv': 'https://docs.google.com/spreadsheets/d/1LDNOSp3p06veKopW0f0JrMZWKNu2G2fM13xyaiJHT3U/edit?gid=1081128644#gid=1081128644',
    'DESI Spectra.csv': 'https://docs.google.com/spreadsheets/d/1xSAIQ_G0XSGvPO1sz9tLGoBZ_oHQOShDYVhyatCpFEQ/edit?gid=1613486317#gid=1613486317',
    'onqueue_classifications.csv': 'https://docs.google.com/spreadsheets/d/1FPBLH8AMnx3cusz2dJXqBcPBnEiIO6ZFKYdSiDpMHok/edit?gid=0#gid=0',
}
GOOGLE_CREDENTIALS_FILE = None  # e.g. Path('/path/to/private/google-credentials.json')
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']

GLOBAL_DATA_DIR = Path('/global/cfs/cdirs/desi/users/zixinq/data')
OUTPUT_ROOT = GLOBAL_DATA_DIR / 'transient_catalog'
SOURCE_DIR = Path('/pscratch/sd/z/zixinq/real_transients')
SELECTION_DIR = OUTPUT_ROOT / 'selection'
MAPPED_MANIFEST = OUTPUT_ROOT / 'selected_transients.csv'
SPECTRA_DIR = OUTPUT_ROOT / 'transient_spectra'
RUN_EXTRACTION = True
print('Run output:', OUTPUT_ROOT)

Run output: /global/cfs/cdirs/desi/users/zixinq/data/transient_catalog


## 2. Selection rules

Use `type`, `TYPE`, or `CHOSEN_CLASS`, depending on the source. Automatic template fits, Redrock classifications, and classifier scores are never treated as truth.

- Retain secure SN, TDE, and afterglow labels. Keep uncertain, blank, and unknown labels in `review.csv`.
- Exclude host/galaxy, AGN/QSO, star, and bad-spectrum labels.
- Preserve exact integer TARGETIDs as strings; quarantine scientific-notation IDs rather than rounding them.
- Combine compatible records with the same complete observation identity. Keep distinct observing epochs.
- Broad-class choices: IIb → SN II; Ic-SLSN → SN Ib/c; Ca-Ia → SN Ia. Preserve every original label.

Edit `LABELS` below if you need a different taxonomy. No extra SNR, phase, or redshift cut is imposed. The DTS file already has an upstream SNR selection.

In [2]:
SOURCES = {
    'prelim_final_table.csv': 'type',
    'dts_classifications_snr5.csv': 'CHOSEN_CLASS',
    'DESI Spectra.csv': 'TYPE',
    'onqueue_classifications.csv': 'CHOSEN_CLASS',
}
LABELS = {
    'SN Ia': {'ia', 'ia-norm', 'ia-91t', 'ia-pec', 'ia-csm', 'ca-ia'},
    'SN II': {'ii', 'ii-norm', 'iin', 'iib', 'ii-slsn'},
    'SN Ib/c': {'ib/c', 'ib-norm', 'ic', 'ic-pec', 'ic-bl', 'ic-slsn'},
    'TDE': {'tde'},
    'afterglow': {'afterglow'},
}
NONTRANSIENT = {'host', 'gal', 'galaxy', 'agn', 'qso', 'quasar', 'star',
                'gwgal', 'gwagn', 'bad', 'blank', 'flat'}


In [3]:
def classify(label):
    label = label.strip().lower()
    for broad, values in LABELS.items():
        if label in values:
            return 'selected', broad
    if label in NONTRANSIENT:
        return 'excluded', ''
    return 'review', ''

def integer(value, optional=False):
    value = str(value).strip()
    if not value and optional:
        return ''
    # Never pass TARGETID through float (DESI IDs exceed float's exact range).
    if not value.isdigit():
        raise ValueError(f'Invalid integer identifier: {value!r}')
    return str(int(value))

def write_csv(path, rows, fields=None):
    fields = fields or list(rows[0])
    with path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8-sig') as handle:
        return list(csv.DictReader(handle))

### Selection implementation
This cell defines the selection and provenance writer; the next cell runs it.

In [4]:
def select(source_dir, output):
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f'Output directory is not empty: {output}')
    rows, hashes = [], {}
    for name, label_column in SOURCES.items():
        path = source_dir / name
        hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()
        for line, original in enumerate(read_csv(path), 2):
            raw_label = original[label_column].strip()
            decision, label = classify(raw_label)
            row = dict(record_id=f'{Path(name).stem}:{line}', source=name,
                       source_row=line, label_column=label_column,
                       raw_label=raw_label, label=label, decision=decision,
                       reason={'selected': 'explicit_transient_label',
                               'excluded': 'nontransient_or_bad_label',
                               'review': 'uncertain_blank_or_unmapped_label'}[decision])
            for key in ['TARGETID', 'NIGHT', 'TILEID', 'PETAL_LOC']:
                try:
                    row[key] = integer(original.get(key, ''), optional=key != 'TARGETID')
                except ValueError:
                    row[key] = original.get(key, '')
                    row['decision'], row['reason'] = 'review', 'invalid_or_precision_lost_identifier'
            row.update(SPECNAME=original.get('SPECNAME', ''),
                       RA=original.get('RA', ''), DEC=original.get('DEC', ''),
                       object_name=original.get('OBJ_ID') or original.get('name', ''),
                       provenance_json=json.dumps(original, sort_keys=True))
            rows.append(row)

    # Only a complete observation identity can be deduplicated at this stage.
    groups = defaultdict(list)
    for row in rows:
        key = tuple(row[k] for k in ['TARGETID', 'NIGHT', 'TILEID', 'PETAL_LOC'])
        groups[key if all(key) else (row['record_id'],)].append(row)
    selected, conflicts = [], []
    for group in groups.values():
        positives = [r for r in group if r['decision'] == 'selected']
        if not positives:
            continue
        labels = {r['label'] for r in positives}
        # A confident negative or incompatible tentative label also requires review.
        incompatible = any(
            r['decision'] == 'excluded' or
            (r['decision'] == 'review' and r['raw_label'].strip('? ').lower() and
             classify(r['raw_label'].strip('? '))[1] not in labels)
            for r in group)
        if len(labels) > 1 or incompatible:
            for row in group:
                row['decision'], row['reason'] = 'review', 'conflicting_observation_labels'
                conflicts.append(row)
            continue
        chosen = dict(positives[0])
        chosen['source_records_json'] = json.dumps([r['record_id'] for r in group])
        chosen['raw_labels_json'] = json.dumps([r['raw_label'] for r in group])
        chosen['spectrum_path'] = ''
        chosen['spectrum_row'] = ''
        chosen['mapping_note'] = ''
        chosen['candidate_cumulative_path'] = ''
        if all(chosen[k] for k in ['NIGHT', 'TILEID', 'PETAL_LOC']):
            t, n, p = (chosen[k] for k in ['TILEID', 'NIGHT', 'PETAL_LOC'])
            chosen['candidate_cumulative_path'] = (
                f'/global/cfs/cdirs/desi/spectro/redux/daily/tiles/cumulative/'
                f'{t}/{n}/coadd-{p}-{t}-thru{n}.fits')
        selected.append(chosen)

    output.mkdir(parents=True, exist_ok=True)
    write_csv(output / 'all_source_rows.csv', rows)
    write_csv(output / 'selected_transients.csv', selected)
    write_csv(output / 'review.csv', [r for r in rows if r['decision'] == 'review'], list(rows[0]))
    write_csv(output / 'excluded.csv', [r for r in rows if r['decision'] == 'excluded'], list(rows[0]))
    summary = dict(
        input_rows=len(rows), selected_observation_records=len(selected),
        selected_unique_targetids=len({r['TARGETID'] for r in selected}),
        selected_by_class=dict(Counter(r['label'] for r in selected)),
        source_decisions={name: dict(Counter(r['decision'] for r in rows if r['source'] == name))
                          for name in SOURCES},
        conflict_source_rows=len(conflicts),
        duplicate_positive_records_collapsed=sum(r['decision'] == 'selected' for r in rows)-len(selected),
        selected_missing_night=sum(not r['NIGHT'] for r in selected),
        selected_missing_tile_or_petal=sum(not r['TILEID'] or not r['PETAL_LOC'] for r in selected),
        spectra_exported=0, source_directory=str(source_dir.resolve()), source_sha256=hashes,
        assumptions=['Observed spectra with secure selected transient labels; no host subtraction.',
                     'SN II includes IIb; SN Ib/c includes Ic-SLSN; Ca-Ia retained under SN Ia. Raw labels preserved.',
                     'TDE and afterglow retained as separate classes.',
                     'AGN/QSO variability not included without an explicit transient label.',
                     'No use of automatic template fits, model scores, or Redrock type as truth.',
                     'No SNR, redshift, phase or spectral-quality cut added.',
                     'Source rows are not proven unique spectra until exact paths and rows are mapped.',
                     'Cumulative path candidates are suggestions from existing project code, not verified mappings.',
                     'Onqueue TILEID is preserved literally; its meaning is not inferred from SPECNAME.',
                     'DIFF is preserved as source metadata, not treated as a host-subtraction flag.'])
    (output / 'selection_summary.json').write_text(json.dumps(summary, indent=2)+'\n')
    print(json.dumps(summary, indent=2))

## 3. Run selection
Input hashes and the label policy are checked before reusing existing results.

In [5]:
missing = [name for name in SOURCES if not (SOURCE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Update SOURCE_DIR; missing CSVs: {missing}')

policy = {'sources': SOURCES, 'labels': {k: sorted(v) for k, v in LABELS.items()},
          'nontransient': sorted(NONTRANSIENT)}
policy_path = SELECTION_DIR / 'notebook_policy.json'
summary_path = SELECTION_DIR / 'selection_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    current_hashes = {name: hashlib.sha256((SOURCE_DIR / name).read_bytes()).hexdigest()
                      for name in SOURCES}
    if current_hashes != summary['source_sha256'] or not policy_path.exists() or json.loads(policy_path.read_text()) != policy:
        raise ValueError('Inputs or label rules changed. Choose a new SELECTION_DIR and rerun.')
    print('Reusing selection with matching input hashes and label rules.')
else:
    with contextlib.redirect_stdout(io.StringIO()):
        select(SOURCE_DIR, SELECTION_DIR)
    policy_path.write_text(json.dumps(policy, indent=2) + '\n')
    summary = json.loads(summary_path.read_text())
    print('Saved selection:', SELECTION_DIR)

# dtype=str is essential: do not load TARGETID through floating point.
selected = pd.read_csv(SELECTION_DIR / 'selected_transients.csv', dtype=str, keep_default_na=False)
assert len(selected) == summary['selected_observation_records']
assert selected['record_id'].is_unique
print(f'{len(selected):,} selected records; {selected.TARGETID.nunique():,} TARGETIDs')
display(selected.groupby('label', sort=True).size().rename('Selected records').to_frame())

Reusing selection with matching input hashes and label rules.
726 selected records; 683 TARGETIDs


,Selected records
label,
SN II,130
SN Ia,543
SN Ib/c,44
TDE,5
afterglow,4


## 5. Map the exact spectra

`prelim_final_table` lacks tile and petal identifiers. `onqueue_classifications` lacks NIGHT and PETAL_LOC: recover the original spectrum manifest using SPECNAME; do not infer its file from TARGETID alone or assume its TILEID column is a HEALPIX.

The candidate cumulative paths below come from the existing project code and **are not verified mappings**. A cumulative coadd may mix observing epochs. Use the actual spectrum that was classified. FAFN is a fiber-assignment file, not a flux spectrum.

Populate `SPECTRUM_MAPPINGS` below by record ID, or edit the generated `mapped_transients.csv` externally, preserving IDs as text. A mapping needs `spectrum_path` and `mapping_note`; `spectrum_row` is the zero-based row of the original FITS file, required only when TARGETID is repeated. Existing mappings are preserved on reruns.

In [6]:
mapping_gaps = selected.assign(
    missing_night=selected.NIGHT.eq(''),
    missing_tile_or_petal=selected.TILEID.eq('') | selected.PETAL_LOC.eq(''))
display(mapping_gaps.groupby('source')[['missing_night', 'missing_tile_or_petal']].sum())
display(selected.loc[selected.candidate_cumulative_path.ne(''),
                     ['record_id', 'candidate_cumulative_path']].head(3))

,missing_night,missing_tile_or_petal
source,,
DESI Spectra.csv,0,0
dts_classifications_snr5.csv,0,0
onqueue_classifications.csv,187,187
prelim_final_table.csv,0,208


,record_id,candidate_cumulative_path
208,dts_classifications_snr5:9,/global/cfs/cdirs/desi/spectro/redux/daily/til...
209,dts_classifications_snr5:10,/global/cfs/cdirs/desi/spectro/redux/daily/til...
210,dts_classifications_snr5:11,/global/cfs/cdirs/desi/spectro/redux/daily/til...


In [10]:
# Replace the commented example with verified mappings for your records.
SPECTRUM_MAPPINGS = {
    # 'dts_classifications_snr5:123': {
    #     'spectrum_path': '/global/.../the_classified_coadd.fits',
    #     'spectrum_row': '',
    #     'mapping_note': 'Matched the original classified-spectrum manifest by SPECNAME.'
    # },
}

mapping_columns = ['spectrum_path', 'spectrum_row', 'mapping_note']
mapped = selected.copy().set_index('record_id', drop=False)
if MAPPED_MANIFEST.exists():
    previous = pd.read_csv(MAPPED_MANIFEST, dtype=str, keep_default_na=False)
    if not previous.record_id.is_unique or set(previous.record_id) != set(mapped.index):
        raise ValueError('Existing mapping IDs differ from selection; choose a new MAPPED_MANIFEST.')
    previous = previous.set_index('record_id')
    # Reuse only mappings; labels and provenance come from the current selection.
    mapped.loc[:, mapping_columns] = previous.loc[mapped.index, mapping_columns]
for record_id, values in SPECTRUM_MAPPINGS.items():
    if record_id not in mapped.index:
        raise KeyError(f'Unknown record_id: {record_id}')
    if set(values) - set(mapping_columns):
        raise ValueError('Only spectrum_path, spectrum_row and mapping_note may be mapped.')
    for column, value in values.items():
        mapped.at[record_id, column] = str(value)
MAPPED_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
mapped.to_csv(MAPPED_MANIFEST, index=False)
ready = mapped.spectrum_path.str.strip().ne('') & mapped.mapping_note.str.strip().ne('')
print(f'{ready.sum():,}/{len(mapped):,} records mapped. Saved: {MAPPED_MANIFEST}')
display(mapped.loc[~ready, ['record_id', 'TARGETID', 'SPECNAME', 'NIGHT']].head(8))

331/726 records mapped. Saved: /global/cfs/cdirs/desi/users/zixinq/data/transient_catalog/selected_transients.csv


,record_id,TARGETID,SPECNAME,NIGHT
record_id,,,,
prelim_final_table:2,prelim_final_table:2,43977521691448588,,20240229
prelim_final_table:3,prelim_final_table:3,39628427841507578,,20240404
prelim_final_table:4,prelim_final_table:4,39628443909884317,,20240404
prelim_final_table:5,prelim_final_table:5,39633519898528592,,20240321
prelim_final_table:6,prelim_final_table:6,39628389291656519,,20240217
prelim_final_table:7,prelim_final_table:7,43977054395651143,,20240129
prelim_final_table:8,prelim_final_table:8,39633038593753508,,20240302
prelim_final_table:9,prelim_final_table:9,2843024463757312,,20240403


## 6. Export spectra

Run with `RUN_EXTRACTION = True` in the parameter cell after mapping. The exporter writes one compressed NPZ per exact source file/row, retaining native bands, wavelength, flux, inverse variance, mask, valid-pixel flags, labels and provenance. No host subtraction, smoothing, normalization, or resampling is performed.

The export records unresolved rows and errors rather than silently dropping them. It may save a partial dataset: check the audit. Repeated TARGETIDs require an explicit row when ambiguous. A new output directory is required for each extraction run.

In [13]:
def spectrum_arrays(spec, targetid, row_index=''):
    import numpy as np
    ids = np.asarray(spec.fibermap['TARGETID'], dtype=np.int64)
    matches = np.flatnonzero(ids == int(targetid))
    if row_index != '':
        index = int(row_index)
        if index < 0 or index >= len(ids) or ids[index] != int(targetid):
            raise ValueError('Mapped spectrum_row does not have the requested TARGETID')
    elif len(matches) == 1:
        index = int(matches[0])
    else:
        raise ValueError(f'Expected one target row; found {len(matches)}. Map spectrum_row explicitly.')
    arrays = {}
    for band in spec.bands:
        wave = np.asarray(spec.wave[band])
        flux = np.asarray(spec.flux[band][index])
        ivar = np.asarray(spec.ivar[band][index])
        mask = np.asarray(spec.mask[band][index])
        if wave.ndim != 1 or not (wave.shape == flux.shape == ivar.shape == mask.shape):
            raise ValueError(f'Invalid array alignment for band {band}')
        if not np.all(np.isfinite(wave)) or not np.all(np.diff(wave) > 0):
            raise ValueError('Wavelength must be finite and strictly increasing')
        valid = np.isfinite(flux) & np.isfinite(ivar) & (ivar > 0) & (mask == 0)
        for key, value in dict(wave=wave, flux=flux, ivar=ivar, mask=mask, valid=valid).items():
            arrays[f'{band}_{key}'] = value
    if not arrays or not any(np.any(v) for k, v in arrays.items() if k.endswith('_valid')):
        raise ValueError('Spectrum has no valid pixels')
    return index, arrays

def extract(manifest, output):
    import numpy as np
    from desispec.io import read_spectra
    rows = read_csv(manifest)
    output.mkdir(parents=True, exist_ok=False)
    mapped = defaultdict(list)
    audit = []
    for row in rows:
        entry = dict(record_id=row['record_id'], TARGETID=row['TARGETID'], label=row['label'],
                     spectrum_path=row['spectrum_path'], spectrum_row=row['spectrum_row'],
                     status='', detail='', output_file='')
        audit.append(entry)
        if not row['spectrum_path'] or not row['mapping_note'].strip():
            entry.update(status='unresolved', detail='Exact spectrum_path and mapping_note are required')
        else:
            mapped[str(Path(row['spectrum_path']).expanduser().resolve())].append((row, entry))
    for path, group in mapped.items():
        try:
            spec = read_spectra(path)
        except Exception as exc:
            for _, entry in group:
                entry.update(status='error', detail=str(exc))
            continue
        by_index = defaultdict(list)
        for row, entry in group:
            try:
                index, arrays = spectrum_arrays(spec, row['TARGETID'], row['spectrum_row'])
                by_index[index].append((row, entry, arrays))
            except Exception as exc:
                entry.update(status='error', detail=str(exc))
        for index, items in by_index.items():
            if len({r['label'] for r, _, _ in items}) != 1:
                for _, entry, _ in items:
                    entry.update(status='conflict', detail='Different labels map to the same spectrum')
                continue
            row, _, arrays = items[0]
            identity = hashlib.sha256(f'{path}:{index}'.encode()).hexdigest()[:24]
            destination = output / f'{identity}.npz'
            # Numeric and Unicode arrays only: load with allow_pickle=False.
            np.savez_compressed(destination, **arrays,
                TARGETID=np.int64(row['TARGETID']), TYPE=np.str_(row['label']),
                source_path=np.str_(path), source_row=np.int64(index),
                metadata_json=np.str_(json.dumps([r for r, _, _ in items], sort_keys=True)))
            with np.load(destination, allow_pickle=False) as check:
                for key, value in arrays.items():
                    if not np.array_equal(check[key], value, equal_nan=True):
                        raise RuntimeError(f'Round-trip mismatch: {key}')
            for j, (_, entry, _) in enumerate(items):
                entry.update(status='saved' if j == 0 else 'duplicate',
                             spectrum_row=index, output_file=str(destination.resolve()))
    write_csv(output / 'extraction_audit.csv', audit)
    counts = dict(Counter(r['status'] for r in audit))
    (output / 'extraction_summary.json').write_text(json.dumps(counts, indent=2)+'\n')
    print(json.dumps(counts, indent=2))
    return 1 if any(r['status'] not in {'saved', 'duplicate'} for r in audit) else 0

In [12]:
has_candidate = selected["candidate_cumulative_path"].str.strip().ne("")
candidates = selected.loc[has_candidate].copy()

candidates["file_exists"] = candidates["candidate_cumulative_path"].map(
    lambda p: Path(p).is_file()
)

display(
    candidates.groupby("source").agg(
        candidates=("TARGETID", "size"),
        files_found=("file_exists", "sum"),
    )
)

,candidates,files_found
source,,
DESI Spectra.csv,29,29
dts_classifications_snr5.csv,302,302


In [13]:
import numpy as np
from astropy.io import fits

mapped = pd.read_csv(
    MAPPED_MANIFEST, dtype=str, keep_default_na=False
).set_index("record_id", drop=False)

issues = []

for path, group in candidates.groupby("candidate_cumulative_path"):
    try:
        with fits.open(path, memmap=True) as hdus:
            ids = np.asarray(
                hdus["FIBERMAP"].data["TARGETID"], dtype=np.int64
            ).copy()

        for _, row in group.iterrows():
            record_id = row["record_id"]

            # Preserve any mapping you have already supplied.
            if mapped.at[record_id, "spectrum_path"].strip():
                continue

            matches = np.flatnonzero(ids == int(row["TARGETID"]))

            if len(matches) != 1:
                issues.append({
                    "record_id": record_id,
                    "reason": f"Found {len(matches)} matching spectrum rows",
                })
                continue

            mapped.at[record_id, "spectrum_path"] = path
            mapped.at[record_id, "spectrum_row"] = str(int(matches[0]))
            mapped.at[record_id, "mapping_note"] = (
                "Cumulative coadd path constructed from catalog "
                "TILEID/NIGHT/PETAL_LOC; unique TARGETID verified in FIBERMAP."
            )

    except Exception as error:
        issues.extend(
            {"record_id": rid, "reason": str(error)}
            for rid in group["record_id"]
        )

mapped.to_csv(MAPPED_MANIFEST, index=False)

ready = (
    mapped["spectrum_path"].str.strip().ne("")
    & mapped["mapping_note"].str.strip().ne("")
)

print(f"{ready.sum()} of {len(mapped)} records mapped")
display(pd.DataFrame(issues, columns=["record_id", "reason"]))

331 of 726 records mapped


,record_id,reason


In [14]:
if not RUN_EXTRACTION:
    print('Extraction not run. Set RUN_EXTRACTION=True after mapping exact spectra.')
elif not ready.any():
    raise ValueError('No mapped spectra. Fill spectrum_path and mapping_note in step 5 first.')
else:
    import importlib.util
    missing_packages = [name for name in ['numpy', 'desispec'] if importlib.util.find_spec(name) is None]
    if missing_packages:
        raise ImportError(f'Select a DESI kernel with: {missing_packages}')
    export_status = extract(MAPPED_MANIFEST, SPECTRA_DIR)
    audit = pd.read_csv(SPECTRA_DIR / 'extraction_audit.csv', dtype=str, keep_default_na=False)
    display(audit.groupby('status').size().rename('Records').to_frame())
    if export_status:
        display(audit.loc[~audit.status.isin(['saved', 'duplicate'])].head(10))
        print('Partial export: resolve failed/unmapped records before treating the dataset as complete.')

INFO:spectra.py:469:read_spectra: iotime 1.150 sec to read spectra from:  coadd-6-83043-thru20240229.fits at 2026-10-15T13:23:45.798496
INFO:spectra.py:469:read_spectra: iotime 1.723 sec to read spectra from:  coadd-1-21183-thru20240404.fits at 2026-10-15T13:23:47.588526
INFO:spectra.py:469:read_spectra: iotime 1.958 sec to read spectra from:  coadd-7-21183-thru20240404.fits at 2026-10-15T13:23:49.649665
INFO:spectra.py:469:read_spectra: iotime 1.735 sec to read spectra from:  coadd-4-23182-thru20240321.fits at 2026-10-15T13:23:51.492185
INFO:spectra.py:469:read_spectra: iotime 1.733 sec to read spectra from:  coadd-7-22969-thru20240217.fits at 2026-10-15T13:23:53.346550
INFO:spectra.py:469:read_spectra: iotime 1.123 sec to read spectra from:  coadd-6-10246-thru20240129.fits at 2026-10-15T13:23:54.567047
INFO:spectra.py:469:read_spectra: iotime 1.655 sec to read spectra from:  coadd-6-24094-thru20240302.fits at 2026-10-15T13:23:56.325082
INFO:spectra.py:469:read_spectra: iotime 2.056 s

,Records
status,
saved,716
unresolved,10


,record_id,TARGETID,label,spectrum_path,spectrum_row,status,detail,output_file
610,onqueue_classifications:152,39628462259961871,SN Ia,,,unresolved,Exact spectrum_path and mapping_note are required,
625,onqueue_classifications:177,39633067576394554,SN Ia,,,unresolved,Exact spectrum_path and mapping_note are required,
665,onqueue_classifications:241,43977405588939296,SN Ia,,,unresolved,Exact spectrum_path and mapping_note are required,
667,onqueue_classifications:243,43977413054816295,SN II,,,unresolved,Exact spectrum_path and mapping_note are required,
671,onqueue_classifications:259,43977456184842801,SN II,,,unresolved,Exact spectrum_path and mapping_note are required,
688,onqueue_classifications:301,43977543820617190,SN Ib/c,,,unresolved,Exact spectrum_path and mapping_note are required,
696,onqueue_classifications:321,43977831151426821,SN II,,,unresolved,Exact spectrum_path and mapping_note are required,
706,onqueue_classifications:335,43978502223285767,SN Ia,,,unresolved,Exact spectrum_path and mapping_note are required,
713,onqueue_classifications:343,43978566924618250,SN II,,,unresolved,Exact spectrum_path and mapping_note are required,
717,onqueue_classifications:349,43978602513290299,SN Ia,,,unresolved,Exact spectrum_path and mapping_note are required,


Partial export: resolve failed/unmapped records before treating the dataset as complete.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from astropy.table import Table
from desispec.spectra import Spectra
from desispec.coaddition import coadd_cameras

DATA_ROOT = Path(
    "/global/cfs/cdirs/desi/users/zixinq/data/transient_catalog"
)

INPUT_DIR = DATA_ROOT / "spectra_coadd_716"  # Existing NPZ export
COADD_DIR = DATA_ROOT / "spectra_coadd_716_npz_combined"

# New directory: keep the original files intact.
audit = pd.read_csv(
    INPUT_DIR / "extraction_audit.csv",
    dtype=str,
    keep_default_na=False,
)
COADD_DIR.mkdir(parents=True, exist_ok=False)

saved_indices = audit.index[audit["status"].eq("saved")]
assert len(saved_indices), "No saved spectra in the extraction audit"

for count, idx in enumerate(saved_indices, start=1):
    row = audit.loc[idx]
    source = INPUT_DIR / Path(row["output_file"]).name
    destination = COADD_DIR / source.name

    try:
        with np.load(source, allow_pickle=False) as raw:
            arrays = {key: raw[key].copy() for key in raw.files}

        if int(arrays["TARGETID"]) != int(row["TARGETID"]):
            raise ValueError("TARGETID differs from extraction audit")
        if str(arrays["TYPE"]) != row["label"]:
            raise ValueError("Label differs from extraction audit")

        # Some exports already have a combined brz band.
        bands = (
            ["brz"] if "brz_wave" in arrays
            else ["b", "r", "z"]
        )

        wave, flux, ivar, mask = {}, {}, {}, {}

        for band in bands:
            required = [
                f"{band}_{name}"
                for name in ("wave", "flux", "ivar", "mask")
            ]
            missing = set(required) - arrays.keys()
            if missing:
                raise ValueError(f"Missing native arrays: {sorted(missing)}")

            w = np.asarray(arrays[f"{band}_wave"], dtype=float)
            f = np.asarray(arrays[f"{band}_flux"], dtype=float)
            v = np.asarray(arrays[f"{band}_ivar"], dtype=float)
            m = np.asarray(arrays[f"{band}_mask"]).copy()

            if w.ndim != 1 or not (w.shape == f.shape == v.shape == m.shape):
                raise ValueError(f"{band}: expected aligned 1D arrays")
            if not np.all(np.isfinite(w)) or not np.all(np.diff(w) > 0):
                raise ValueError(f"{band}: invalid wavelength grid")

            valid = np.isfinite(f) & np.isfinite(v) & (v > 0) & (m == 0)
            if f"{band}_valid" in arrays:
                valid &= arrays[f"{band}_valid"].astype(bool)

            # Exclude invalid pixels from camera combination.
            wave[band] = w
            flux[band] = np.where(valid, f, 0.0)[None, :]
            ivar[band] = np.where(valid, v, 0.0)[None, :]
            mask[band] = m[None, :]

        spectrum = Spectra(
            bands=bands,
            wave=wave,
            flux=flux,
            ivar=ivar,
            mask=mask,
            fibermap=Table({
                "TARGETID": np.array([int(arrays["TARGETID"])], dtype=np.int64)
            }),
        )

        combined = (
            spectrum if bands == ["brz"]
            else coadd_cameras(spectrum)
        )

        arrays["coadd_wave"] = combined.wave["brz"]
        arrays["coadd_flux"] = combined.flux["brz"][0]
        arrays["coadd_ivar"] = combined.ivar["brz"][0]
        arrays["coadd_mask"] = combined.mask["brz"][0]
        arrays["coadd_valid"] = (
            np.isfinite(arrays["coadd_flux"])
            & np.isfinite(arrays["coadd_ivar"])
            & (arrays["coadd_ivar"] > 0)
            & (arrays["coadd_mask"] == 0)
        )
        arrays["camera_coadd_input"] = np.str_(str(source.resolve()))

        # Preserve native arrays, labels, and original provenance.
        np.savez_compressed(destination, **arrays)
        audit.loc[idx, "output_file"] = str(destination.resolve())

    except Exception as exc:
        audit.loc[idx, "status"] = "error"
        audit.loc[idx, "detail"] = f"NPZ camera coaddition: {exc}"
        audit.loc[idx, "output_file"] = ""

    if count % 100 == 0 or count == len(saved_indices):
        print(f"Processed {count}/{len(saved_indices)}")

# Keep duplicate audit pointers consistent with the new directory.
for idx in audit.index[audit["status"].eq("duplicate")]:
    old_path = audit.loc[idx, "output_file"]
    if old_path:
        new_path = COADD_DIR / Path(old_path).name
        audit.loc[idx, "output_file"] = (
            str(new_path.resolve()) if new_path.exists() else ""
        )

audit.to_csv(COADD_DIR / "extraction_audit.csv", index=False)
summary = audit["status"].value_counts().to_dict()
(COADD_DIR / "extraction_summary.json").write_text(
    json.dumps(summary, indent=2)
)

display(audit["status"].value_counts().rename("Records"))
display(
    audit.loc[audit["status"].eq("error"), ["record_id", "detail"]]
)
print("Saved to:", COADD_DIR)

## 7. Load one saved spectrum

Use `allow_pickle=False`. Each camera has its own wavelength grid. Apply the same preprocessing as your classifier before training. TDE and afterglow need separate classes or exclusion if your classifier only supports Ia/II/Ib-c. Split by astrophysical object across epochs and aliases; this notebook does not create train/test splits.

In [34]:
if RUN_EXTRACTION and (SPECTRA_DIR / 'extraction_audit.csv').exists():
    import numpy as np
    saved = audit.loc[audit.status == 'saved', 'output_file']
    if len(saved):
        with np.load(saved.iloc[0], allow_pickle=False) as spectrum:
            print('TARGETID:', int(spectrum['TARGETID']), '| label:', str(spectrum['TYPE']))
            print('Arrays:', spectrum.files)
            bands = [key[:-5] for key in spectrum.files if key.endswith('_wave')]
            display(pd.DataFrame([
                {'band': band, 'pixels': len(spectrum[f'{band}_wave']),
                 'valid_pixels': int(spectrum[f'{band}_valid'].sum())}
                for band in bands]))
    else:
        print('No spectra were saved; inspect the extraction audit.')
else:
    print('No spectrum loaded: extraction has not run in this notebook session.')

TARGETID: 43976918173072101 | label: SN II
Arrays: ['b_wave', 'b_flux', 'b_ivar', 'b_mask', 'b_valid', 'r_wave', 'r_flux', 'r_ivar', 'r_mask', 'r_valid', 'z_wave', 'z_flux', 'z_ivar', 'z_mask', 'z_valid', 'TARGETID', 'TYPE', 'source_path', 'source_row', 'metadata_json']


,band,pixels,valid_pixels
0,b,2751,2739
1,r,2326,2305
2,z,2881,2843


In [17]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from astropy.table import Table
from desispec.spectra import Spectra
from desispec.coaddition import coadd_cameras

DATA_ROOT = Path(
    "/global/cfs/cdirs/desi/users/zixinq/data/transient_catalog"
)

INPUT_DIR = DATA_ROOT / "transient_spectra"  # Existing NPZ export
COADD_DIR = DATA_ROOT / "transient_spectra_coadd"

# New directory: keep the original files intact.
audit = pd.read_csv(
    INPUT_DIR / "extraction_audit.csv",
    dtype=str,
    keep_default_na=False,
)
COADD_DIR.mkdir(parents=True, exist_ok=False)

saved_indices = audit.index[audit["status"].eq("saved")]
assert len(saved_indices), "No saved spectra in the extraction audit"

for count, idx in enumerate(saved_indices, start=1):
    row = audit.loc[idx]
    source = INPUT_DIR / Path(row["output_file"]).name
    destination = COADD_DIR / source.name

    try:
        with np.load(source, allow_pickle=False) as raw:
            arrays = {key: raw[key].copy() for key in raw.files}

        if int(arrays["TARGETID"]) != int(row["TARGETID"]):
            raise ValueError("TARGETID differs from extraction audit")
        if str(arrays["TYPE"]) != row["label"]:
            raise ValueError("Label differs from extraction audit")

        # Some exports already have a combined brz band.
        bands = (
            ["brz"] if "brz_wave" in arrays
            else ["b", "r", "z"]
        )

        wave, flux, ivar, mask = {}, {}, {}, {}

        for band in bands:
            required = [
                f"{band}_{name}"
                for name in ("wave", "flux", "ivar", "mask")
            ]
            missing = set(required) - arrays.keys()
            if missing:
                raise ValueError(f"Missing native arrays: {sorted(missing)}")

            w = np.asarray(arrays[f"{band}_wave"], dtype=float)
            f = np.asarray(arrays[f"{band}_flux"], dtype=float)
            v = np.asarray(arrays[f"{band}_ivar"], dtype=float)
            m = np.asarray(arrays[f"{band}_mask"]).copy()

            if w.ndim != 1 or not (w.shape == f.shape == v.shape == m.shape):
                raise ValueError(f"{band}: expected aligned 1D arrays")
            if not np.all(np.isfinite(w)) or not np.all(np.diff(w) > 0):
                raise ValueError(f"{band}: invalid wavelength grid")

            valid = np.isfinite(f) & np.isfinite(v) & (v > 0) & (m == 0)
            if f"{band}_valid" in arrays:
                valid &= arrays[f"{band}_valid"].astype(bool)

            # Exclude invalid pixels from camera combination.
            wave[band] = w
            flux[band] = np.where(valid, f, 0.0)[None, :]
            ivar[band] = np.where(valid, v, 0.0)[None, :]
            mask[band] = m[None, :]

        spectrum = Spectra(
            bands=bands,
            wave=wave,
            flux=flux,
            ivar=ivar,
            mask=mask,
            fibermap=Table({
                "TARGETID": np.array([int(arrays["TARGETID"])], dtype=np.int64)
            }),
        )

        combined = (
            spectrum if bands == ["brz"]
            else coadd_cameras(spectrum)
        )

        arrays["coadd_wave"] = combined.wave["brz"]
        arrays["coadd_flux"] = combined.flux["brz"][0]
        arrays["coadd_ivar"] = combined.ivar["brz"][0]
        arrays["coadd_mask"] = combined.mask["brz"][0]
        arrays["coadd_valid"] = (
            np.isfinite(arrays["coadd_flux"])
            & np.isfinite(arrays["coadd_ivar"])
            & (arrays["coadd_ivar"] > 0)
            & (arrays["coadd_mask"] == 0)
        )
        arrays["camera_coadd_input"] = np.str_(str(source.resolve()))

        # Preserve native arrays, labels, and original provenance.
        np.savez_compressed(destination, **arrays)
        audit.loc[idx, "output_file"] = str(destination.resolve())

    except Exception as exc:
        audit.loc[idx, "status"] = "error"
        audit.loc[idx, "detail"] = f"NPZ camera coaddition: {exc}"
        audit.loc[idx, "output_file"] = ""

    if count % 100 == 0 or count == len(saved_indices):
        print(f"Processed {count}/{len(saved_indices)}")

# Keep duplicate audit pointers consistent with the new directory.
for idx in audit.index[audit["status"].eq("duplicate")]:
    old_path = audit.loc[idx, "output_file"]
    if old_path:
        new_path = COADD_DIR / Path(old_path).name
        audit.loc[idx, "output_file"] = (
            str(new_path.resolve()) if new_path.exists() else ""
        )

audit.to_csv(COADD_DIR / "extraction_audit.csv", index=False)
summary = audit["status"].value_counts().to_dict()
(COADD_DIR / "extraction_summary.json").write_text(
    json.dumps(summary, indent=2)
)

display(audit["status"].value_counts().rename("Records"))
display(
    audit.loc[audit["status"].eq("error"), ["record_id", "detail"]]
)
print("Saved to:", COADD_DIR)

INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.095 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.093 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.100 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.098 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.097 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.106 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.105 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.099 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.095 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.094 [sec]
INFO:coaddition.py:1395:coadd_cameras: coadding spectra across cameras took: 0.097 [sec]
INFO:coaddition.py:13

status
saved         716
unresolved     10
Name: Records, dtype: int64

,record_id,detail


Saved to: /global/cfs/cdirs/desi/users/zixinq/data/transient_catalog/transient_spectra_coadd


In [19]:
from pathlib import Path

# Records with no match in the Loa HEALPix catalog
matched_ids = set(onqueue_candidates["record_id"])

remaining = manifest.loc[
    manifest["source"].eq("onqueue_classifications.csv")
    & ~manifest["record_id"].isin(matched_ids)
].copy()

daily_tiles = Path(
    "/global/cfs/cdirs/desi/spectro/redux/daily/tiles/cumulative"
)

def tile_directory(value):
    value = str(value).strip()
    return daily_tiles / str(int(value)) if value.isdigit() else None

remaining["tile_directory"] = remaining["TILEID"].map(tile_directory)
remaining["tile_exists"] = remaining["tile_directory"].map(
    lambda p: p is not None and p.is_dir()
)

print(f"{len(remaining)} unmatched on-queue records")
print(f"{remaining['tile_exists'].sum()} have an existing daily tile directory")

display(
    remaining[
        ["record_id", "TARGETID", "TILEID", "RA", "DEC", "tile_exists"]
    ].head(15)
)

132 unmatched on-queue records
52 have an existing daily tile directory


,record_id,TARGETID,TILEID,RA,DEC,tile_exists
540,onqueue_classifications:4,2707083182997506,27234,192.8293283,40.20148036,True
541,onqueue_classifications:5,2842079726141446,37568,78.64162381,-18.74740572,False
543,onqueue_classifications:9,2843098883293184,36310,14.52494165,24.41982785,False
544,onqueue_classifications:10,2843446159081474,26200,243.4098417,41.54795746,True
545,onqueue_classifications:11,2843604829601795,27779,218.1280379,50.9169827,True
546,onqueue_classifications:12,2843824355278848,28362,195.3296494,69.13003434,False
547,onqueue_classifications:14,3146684347449345,21338,140.9803684,29.88319521,True
548,onqueue_classifications:15,3146954938777601,21955,170.0662433,43.90196345,True
549,onqueue_classifications:16,3155072636682246,35769,331.3497369,11.65951368,False
550,onqueue_classifications:18,3155938320056321,27759,229.0242627,56.01954523,True


In [20]:
import numpy as np
import pandas as pd
import fitsio

hits, errors = [], []

for tile_dir, group in remaining.loc[
    remaining["tile_exists"]
].groupby("tile_directory"):

    tile_dir = Path(tile_dir)
    wanted = {int(t) for t in group["TARGETID"]}

    for path in sorted(tile_dir.glob("*/coadd-*.fits*")):
        try:
            fm = fitsio.read(
                str(path), ext="FIBERMAP", columns=["TARGETID"]
            )
            indices = np.flatnonzero(
                np.isin(fm["TARGETID"], list(wanted))
            )

            for index in indices:
                tid = str(int(fm["TARGETID"][index]))
                for record_id in group.loc[
                    group["TARGETID"].eq(tid), "record_id"
                ]:
                    hits.append({
                        "record_id": record_id,
                        "TARGETID": tid,
                        "candidate_path": str(path),
                        "spectrum_row": int(index),
                        "coadd_through_night": path.parent.name,
                    })

        except Exception as error:
            errors.append({"path": str(path), "reason": str(error)})

daily_matches = pd.DataFrame(hits, columns=[
    "record_id", "TARGETID", "candidate_path",
    "spectrum_row", "coadd_through_night",
])
daily_errors = pd.DataFrame(errors, columns=["path", "reason"])

daily_matches.to_csv(
    OUTPUT_ROOT / "onqueue_daily_tile_matches.csv", index=False
)
daily_errors.to_csv(
    OUTPUT_ROOT / "onqueue_daily_tile_errors.csv", index=False
)

print(
    f"{daily_matches['record_id'].nunique()} of 52 records "
    f"found by exact TARGETID"
)
print(f"{len(daily_matches)} candidate spectrum rows")
print(f"{len(daily_errors)} file-read errors")
display(daily_matches.head(10))

0 of 52 records found by exact TARGETID
0 candidate spectrum rows
0 file-read errors


,record_id,TARGETID,candidate_path,spectrum_row,coadd_through_night


In [21]:
tile_dirs = {
    Path(p)
    for p in remaining.loc[remaining["tile_exists"], "tile_directory"]
}

file_counts = pd.DataFrame([
    {
        "tile_directory": str(p),
        "coadd_files": len(list(p.glob("*/coadd-*.fits*"))),
    }
    for p in sorted(tile_dirs)
])

print("Total candidate files:", file_counts["coadd_files"].sum())
display(file_counts)

Total candidate files: 576


,tile_directory,coadd_files
0,/global/cfs/cdirs/desi/spectro/redux/daily/til...,10
1,/global/cfs/cdirs/desi/spectro/redux/daily/til...,9
2,/global/cfs/cdirs/desi/spectro/redux/daily/til...,20
3,/global/cfs/cdirs/desi/spectro/redux/daily/til...,10
4,/global/cfs/cdirs/desi/spectro/redux/daily/til...,10
5,/global/cfs/cdirs/desi/spectro/redux/daily/til...,10
6,/global/cfs/cdirs/desi/spectro/redux/daily/til...,10
7,/global/cfs/cdirs/desi/spectro/redux/daily/til...,10
8,/global/cfs/cdirs/desi/spectro/redux/daily/til...,10
9,/global/cfs/cdirs/desi/spectro/redux/daily/til...,20


In [22]:
onqueue = pd.read_csv(
    SOURCE_DIR / "onqueue_classifications.csv",
    dtype=str,
    keep_default_na=False,
)
display(onqueue["FAFLAVOR"].value_counts())

FAFLAVOR
mainbright    234
maindark      123
Name: count, dtype: int64

In [23]:
from pathlib import Path

daily = Path("/global/cfs/cdirs/desi/spectro/redux/daily")

inventory_files = sorted({
    p
    for pattern in (
        "exposures*.fits*",
        "tiles*.fits*",
        "exposure_tables/*/*.csv",
        "exposure_tables/*/*.ecsv",
    )
    for p in daily.glob(pattern)
})

for path in inventory_files[:30]:
    print(path)

print(f"\nFound {len(inventory_files)} inventory files")

/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200219.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200220.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200223.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200224.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200225.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200226.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200227.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200228.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202002/exposure_table_20200229.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/202003/exposure_table_20200301.csv
/global/cfs/cdirs/desi/spectro/redux/daily/exposure_tables/2

In [24]:
from astropy.io import fits

exposure_catalogs = sorted(daily.glob("exposures*.fits*"))

if exposure_catalogs:
    for path in exposure_catalogs:
        print(f"\nFILE: {path}")
        with fits.open(path, memmap=True) as hdus:
            for hdu in hdus:
                if isinstance(hdu, fits.BinTableHDU):
                    print("Extension:", hdu.name)
                    print("Rows:", hdu.header.get("NAXIS2"))
                    print("Columns:", ", ".join(hdu.columns.names))
else:
    # Inspect one exposure-table CSV if no combined FITS inventory exists.
    tables = [
        p for p in inventory_files
        if p.suffix == ".csv" and "exposure_tables" in p.parts
    ]
    if tables:
        path = tables[0]
        print("Example exposure table:", path)
        print("Columns:", pd.read_csv(path, nrows=0).columns.tolist())
    else:
        print("First inventory paths:")
        print("\n".join(map(str, inventory_files[:10])))


FILE: /global/cfs/cdirs/desi/spectro/redux/daily/exposures-daily.fits
Extension: EXPOSURES
Rows: 40255
Columns: NIGHT, EXPID, TILEID, TILERA, TILEDEC, MJD, SURVEY, PROGRAM, FAPRGRM, FAFLAVOR, EXPTIME, EFFTIME_SPEC, GOALTIME, GOALTYPE, MINTFRAC, AIRMASS, EBV, SEEING_ETC, EFFTIME_ETC, TSNR2_ELG, TSNR2_QSO, TSNR2_LRG, TSNR2_LYA, TSNR2_BGS, TSNR2_GPBDARK, TSNR2_GPBBRIGHT, TSNR2_GPBBACKUP, LRG_EFFTIME_DARK, ELG_EFFTIME_DARK, BGS_EFFTIME_BRIGHT, LYA_EFFTIME_DARK, GPB_EFFTIME_DARK, GPB_EFFTIME_BRIGHT, GPB_EFFTIME_BACKUP, TRANSPARENCY_GFA, SEEING_GFA, FIBER_FRACFLUX_GFA, FIBER_FRACFLUX_ELG_GFA, FIBER_FRACFLUX_BGS_GFA, FIBERFAC_GFA, FIBERFAC_ELG_GFA, FIBERFAC_BGS_GFA, AIRMASS_GFA, SKY_MAG_AB_GFA, SKY_MAG_G_SPEC, SKY_MAG_R_SPEC, SKY_MAG_Z_SPEC, EFFTIME_GFA, EFFTIME_DARK_GFA, EFFTIME_BRIGHT_GFA, EFFTIME_BACKUP_GFA
Extension: FRAMES
Rows: 1189349
Columns: NIGHT, EXPID, TILEID, EXPTIME, SEEING_ETC, EFFTIME_ETC, CAMERA, TSNR2_ELG, TSNR2_BGS, TSNR2_QSO, TSNR2_LRG, SURVEY, GOALTYPE, FAFLAVOR, MINTFRA

In [25]:
import json
import numpy as np
import pandas as pd
import fitsio
from pathlib import Path

daily = Path("/global/cfs/cdirs/desi/spectro/redux/daily")

TIME_WINDOW_DAYS = 1.5
TILE_RADIUS_DEG = 2.0
POSITION_RADIUS_ARCSEC = 1.0

exposures = fitsio.read(
    str(daily / "exposures-daily.fits"),
    ext="EXPOSURES",
    columns=["NIGHT", "EXPID", "TILEID", "MJD", "TILERA", "TILEDEC"],
)


def separation_deg(ra, dec, other_ra, other_dec):
    """Great-circle separation, stable at small angular distances."""
    ra1, dec1 = np.deg2rad(float(ra)), np.deg2rad(float(dec))
    ra2 = np.deg2rad(np.asarray(other_ra, dtype=float))
    dec2 = np.deg2rad(np.asarray(other_dec, dtype=float))
    a = (
        np.sin((dec2 - dec1) / 2)**2
        + np.cos(dec1) * np.cos(dec2) * np.sin((ra2 - ra1) / 2)**2
    )
    return np.rad2deg(2 * np.arcsin(np.sqrt(np.clip(a, 0, 1))))


hits, search_audit, errors = [], [], []
fibermap_cache = {}

for _, record in remaining.iterrows():
    record_id = record["record_id"]

    try:
        original = json.loads(record["provenance_json"])
        obs_mjd = float(original["OBS_MJD"])
        ra, dec = float(record["RA"]), float(record["DEC"])
        targetid = int(record["TARGETID"])

        if not np.all(np.isfinite([obs_mjd, ra, dec])):
            raise ValueError("Non-finite observation time or coordinates")

        dt = np.abs(exposures["MJD"] - obs_mjd)
        distance = separation_deg(
            ra, dec, exposures["TILERA"], exposures["TILEDEC"]
        )
        nearby = exposures[
            (dt <= TIME_WINDOW_DAYS) & (distance <= TILE_RADIUS_DEG)
        ]

        # Several exposures can point to the same cumulative coadd.
        tile_nights = {
            (int(e["TILEID"]), int(e["NIGHT"])) for e in nearby
        }
        files = set()
        for tile, night in tile_nights:
            directory = daily / "tiles/cumulative" / str(tile) / str(night)
            files.update(directory.glob("coadd-*.fits*"))

        hit_count = 0

        for path in sorted(files):
            try:
                if path not in fibermap_cache:
                    fibermap_cache[path] = fitsio.read(
                        str(path),
                        ext="FIBERMAP",
                        columns=["TARGETID", "TARGET_RA", "TARGET_DEC"],
                    )
                fm = fibermap_cache[path]

                separation = 3600 * separation_deg(
                    ra, dec, fm["TARGET_RA"], fm["TARGET_DEC"]
                )
                exact_id = fm["TARGETID"] == targetid
                close_position = separation <= POSITION_RADIUS_ARCSEC

                for index in np.flatnonzero(exact_id | close_position):
                    hit_count += 1
                    hits.append({
                        "record_id": record_id,
                        "requested_targetid": str(targetid),
                        "found_targetid": str(int(fm["TARGETID"][index])),
                        "exact_targetid": bool(exact_id[index]),
                        "position_match": bool(close_position[index]),
                        "separation_arcsec": float(separation[index]),
                        "OBS_MJD": obs_mjd,
                        "candidate_path": str(path),
                        "spectrum_row": int(index),
                        "coadd_through_night": path.parent.name,
                    })

            except Exception as error:
                errors.append({
                    "record_id": record_id,
                    "path": str(path),
                    "reason": str(error),
                })

        search_audit.append({
            "record_id": record_id,
            "candidate_exposures": len(nearby),
            "candidate_files": len(files),
            "candidate_spectrum_rows": hit_count,
        })

    except Exception as error:
        errors.append({
            "record_id": record_id, "path": "", "reason": str(error)
        })


columns = [
    "record_id", "requested_targetid", "found_targetid",
    "exact_targetid", "position_match", "separation_arcsec",
    "OBS_MJD", "candidate_path", "spectrum_row", "coadd_through_night",
]
onqueue_time_position_matches = pd.DataFrame(hits, columns=columns)
search_audit = pd.DataFrame(search_audit)
errors = pd.DataFrame(errors, columns=["record_id", "path", "reason"])

onqueue_time_position_matches.to_csv(
    OUTPUT_ROOT / "onqueue_time_position_matches.csv", index=False
)
search_audit.to_csv(
    OUTPUT_ROOT / "onqueue_time_position_search_audit.csv", index=False
)
errors.to_csv(
    OUTPUT_ROOT / "onqueue_time_position_errors.csv", index=False
)

both = onqueue_time_position_matches.loc[
    onqueue_time_position_matches["exact_targetid"].eq(True)
    & onqueue_time_position_matches["position_match"].eq(True)
]

print("Records searched:", len(remaining))
print("Records with TARGETID + position matches:", both["record_id"].nunique())
print("Records with any candidate match:",
      onqueue_time_position_matches["record_id"].nunique())
print("Errors:", len(errors))

display(onqueue_time_position_matches.head(15))

Records searched: 132
Records with TARGETID + position matches: 125
Records with any candidate match: 125
Errors: 0


,record_id,requested_targetid,found_targetid,exact_targetid,position_match,separation_arcsec,OBS_MJD,candidate_path,spectrum_row,coadd_through_night
0,onqueue_classifications:4,2707083182997506,2707083182997506,True,True,0.000092,60756.31932,/global/cfs/cdirs/desi/spectro/redux/daily/til...,138,20250321
1,onqueue_classifications:5,2842079726141446,39627340745018451,False,True,0.000021,60968.49354,/global/cfs/cdirs/desi/spectro/redux/daily/til...,326,20251019
2,onqueue_classifications:5,2842079726141446,2842079726141446,True,True,0.000021,60968.49354,/global/cfs/cdirs/desi/spectro/redux/daily/til...,327,20251019
3,onqueue_classifications:9,2843098883293184,2843098883293184,True,True,0.000012,60587.35091,/global/cfs/cdirs/desi/spectro/redux/daily/til...,330,20241003
4,onqueue_classifications:10,2843446159081474,2843446159081474,True,True,0.000092,60500.19450,/global/cfs/cdirs/desi/spectro/redux/daily/til...,94,20240708
5,onqueue_classifications:11,2843604829601795,2843604829601795,True,True,0.000047,60855.17993,/global/cfs/cdirs/desi/spectro/redux/daily/til...,382,20250628
6,onqueue_classifications:12,2843824355278848,2843824355278848,True,True,0.000019,61096.29652,/global/cfs/cdirs/desi/spectro/redux/daily/til...,48,20260224
7,onqueue_classifications:14,3146684347449345,3146684347449345,True,True,0.000060,60671.36559,/global/cfs/cdirs/desi/spectro/redux/daily/til...,70,20241226
8,onqueue_classifications:15,3146954938777601,3146954938777601,True,True,0.000048,60732.29892,/global/cfs/cdirs/desi/spectro/redux/daily/til...,59,20250225
9,onqueue_classifications:16,3155072636682246,3155072636682246,True,True,0.000086,60500.91368,/global/cfs/cdirs/desi/spectro/redux/daily/til...,410,20240708


In [26]:
verified_candidates = onqueue_time_position_matches.loc[
    onqueue_time_position_matches["exact_targetid"].eq(True)
    & onqueue_time_position_matches["position_match"].eq(True)
].drop_duplicates(["record_id", "candidate_path", "spectrum_row"])

counts = verified_candidates.groupby("record_id").size()

print("One candidate:", counts.eq(1).sum())
print("Multiple candidates:", counts.gt(1).sum())

unmatched = remaining.loc[
    ~remaining["record_id"].isin(counts.index)
]
display(unmatched[["record_id", "TARGETID", "RA", "DEC"]])

One candidate: 121
Multiple candidates: 4


,record_id,TARGETID,RA,DEC
625,onqueue_classifications:177,39633067576394554,179.4176623,39.61976113
665,onqueue_classifications:241,43977405588939296,70.379119,-1.985091
671,onqueue_classifications:259,43977456184842801,206.118333,-0.118231
688,onqueue_classifications:301,43977543820617190,30.785125,3.749942
706,onqueue_classifications:335,43978502223285767,175.832031,49.121384
713,onqueue_classifications:343,43978566924618250,177.55456,53.246094
717,onqueue_classifications:349,43978602513290299,216.236367,55.634557


In [27]:
ambiguous_ids = counts.index[counts.gt(1)]

display(
    verified_candidates.loc[
        verified_candidates["record_id"].isin(ambiguous_ids),
        [
            "record_id", "requested_targetid", "OBS_MJD",
            "coadd_through_night", "separation_arcsec",
            "candidate_path", "spectrum_row",
        ],
    ].sort_values(["record_id", "coadd_through_night"])
)

,record_id,requested_targetid,OBS_MJD,coadd_through_night,separation_arcsec,candidate_path,spectrum_row
32,onqueue_classifications:108,39628079005438588,60698.09371,20250121,0.000015,/global/cfs/cdirs/desi/spectro/redux/daily/til...,408
33,onqueue_classifications:108,39628079005438588,60698.09371,20250123,0.000015,/global/cfs/cdirs/desi/spectro/redux/daily/til...,408
44,onqueue_classifications:158,39632930032583931,60628.02370,20241112,0.000041,/global/cfs/cdirs/desi/spectro/redux/daily/til...,123
45,onqueue_classifications:158,39632930032583931,60628.02370,20241113,0.000041,/global/cfs/cdirs/desi/spectro/redux/daily/til...,123
9,onqueue_classifications:16,3155072636682246,60500.91368,20240708,0.000086,/global/cfs/cdirs/desi/spectro/redux/daily/til...,410
10,onqueue_classifications:16,3155072636682246,60500.91368,20240709,0.000086,/global/cfs/cdirs/desi/spectro/redux/daily/til...,410
68,onqueue_classifications:228,43977112751030209,61058.72668,20260117,0.000000,/global/cfs/cdirs/desi/spectro/redux/daily/til...,94
69,onqueue_classifications:228,43977112751030209,61058.72668,20260118,0.000000,/global/cfs/cdirs/desi/spectro/redux/daily/til...,94


In [28]:
import fitsio
import numpy as np
import pandas as pd

checks = []

for _, candidate in verified_candidates.iterrows():
    result = candidate.to_dict()

    try:
        fm = fitsio.read(
            candidate["candidate_path"],
            ext="FIBERMAP",
            rows=[int(candidate["spectrum_row"])],
        )
        row = fm[0]

        if int(row["TARGETID"]) != int(candidate["requested_targetid"]):
            raise ValueError("Spectrum row has a different TARGETID")

        for column in [
            "MEAN_MJD", "MIN_MJD", "MAX_MJD",
            "COADD_NUMEXP", "COADD_NUMNIGHT",
        ]:
            result[column] = (
                row[column] if column in fm.dtype.names else np.nan
            )

        result["mean_time_difference_seconds"] = (
            abs(float(result["MEAN_MJD"]) - float(candidate["OBS_MJD"]))
            * 86400
        )
        result["error"] = ""

    except Exception as error:
        result["error"] = str(error)

    checks.append(result)

coadd_time_checks = pd.DataFrame(checks)
coadd_time_checks.to_csv(
    OUTPUT_ROOT / "onqueue_coadd_time_checks.csv", index=False
)

display(
    coadd_time_checks.loc[
        coadd_time_checks["record_id"].isin(ambiguous_ids),
        [
            "record_id", "coadd_through_night", "OBS_MJD",
            "MEAN_MJD", "MIN_MJD", "MAX_MJD",
            "COADD_NUMNIGHT", "mean_time_difference_seconds", "error",
        ],
    ].sort_values(["record_id", "mean_time_difference_seconds"])
)

,record_id,coadd_through_night,OBS_MJD,MEAN_MJD,MIN_MJD,MAX_MJD,COADD_NUMNIGHT,mean_time_difference_seconds,error
31,onqueue_classifications:108,20250123,60698.09371,60698.093706,60697.093493,60699.093920,2,0.309744,
30,onqueue_classifications:108,20250121,60698.09371,60697.093493,60697.093493,60697.093493,1,86418.775584,
43,onqueue_classifications:158,20241113,60628.02370,60628.023704,60627.539176,60628.508231,2,0.320544,
42,onqueue_classifications:158,20241112,60628.02370,60627.539176,60627.539176,60627.539176,1,41862.841632,
9,onqueue_classifications:16,20240709,60500.91368,60500.913680,60500.406896,60501.420463,2,0.002160,
8,onqueue_classifications:16,20240708,60500.91368,60500.406896,60500.406896,60500.406896,1,43786.097856,
67,onqueue_classifications:228,20260118,61058.72668,61058.726681,61058.248068,61059.205294,2,0.058320,
66,onqueue_classifications:228,20260117,61058.72668,61058.248068,61058.248068,61058.248068,1,41352.091488,


In [29]:
checks = coadd_time_checks.copy()
checks["mean_time_difference_seconds"] = pd.to_numeric(
    checks["mean_time_difference_seconds"], errors="coerce"
)

passing = checks.loc[
    checks["error"].fillna("").eq("")
    & checks["mean_time_difference_seconds"].le(1.0)
].drop_duplicates(["record_id", "candidate_path", "spectrum_row"])

n_matches = passing.groupby("record_id").size()
accepted = passing.loc[
    passing["record_id"].isin(n_matches.index[n_matches.eq(1)])
]

mapped = pd.read_csv(
    MAPPED_MANIFEST, dtype=str, keep_default_na=False
).set_index("record_id", drop=False)

added = 0
for _, row in accepted.iterrows():
    rid = row["record_id"]

    # Preserve existing mappings.
    if mapped.at[rid, "spectrum_path"].strip():
        continue

    mapped.at[rid, "spectrum_path"] = row["candidate_path"]
    mapped.at[rid, "spectrum_row"] = str(int(row["spectrum_row"]))
    mapped.at[rid, "mapping_note"] = (
        "Exact TARGETID and position verified; unique candidate "
        "with coadd MEAN_MJD matching classified OBS_MJD within 1 second."
    )
    added += 1

mapped.to_csv(MAPPED_MANIFEST, index=False)

ready = (
    mapped["spectrum_path"].str.strip().ne("")
    & mapped["mapping_note"].str.strip().ne("")
)

print(f"Added {added} mappings")
print(f"{ready.sum()} of {len(mapped)} records mapped")

Added 123 mappings
454 of 726 records mapped


In [30]:
SPECTRA_DIR = OUTPUT_ROOT / "transient_spectra"
RUN_EXTRACTION = True

In [7]:
import fitsio
import numpy as np
import pandas as pd

prelim = pd.read_csv(
    OUTPUT_ROOT / "prelim_final_table_candidates.csv",
    dtype=str, keep_default_na=False,
)

mapped = pd.read_csv(
    MAPPED_MANIFEST, dtype=str, keep_default_na=False
).set_index("record_id", drop=False)

verified, issues = [], []

for path, group in prelim.groupby("lookup_path"):
    try:
        fm = fitsio.read(
            path, ext="FIBERMAP", columns=["TARGETID"]
        )
        exposures = fitsio.read(
            path, ext="EXP_FIBERMAP",
            columns=["TARGETID", "NIGHT"],
        )

        for _, row in group.iterrows():
            rid = row["record_id"]
            if mapped.at[rid, "spectrum_path"].strip():
                continue

            tid = int(row["TARGETID"])
            night = int(row["NIGHT"])

            indices = np.flatnonzero(fm["TARGETID"] == tid)
            observed_on_night = np.any(
                (exposures["TARGETID"] == tid)
                & (exposures["NIGHT"] == night)
            )

            if len(indices) == 1 and observed_on_night:
                verified.append({
                    "record_id": rid,
                    "spectrum_path": path,
                    "spectrum_row": str(int(indices[0])),
                })
            else:
                issues.append({
                    "record_id": rid,
                    "path": path,
                    "reason": (
                        f"{len(indices)} target rows; "
                        f"exposure on requested night={observed_on_night}"
                    ),
                })

    except Exception as error:
        issues.extend({
            "record_id": rid, "path": path, "reason": str(error)
        } for rid in group["record_id"])

verified = pd.DataFrame(verified, columns=[
    "record_id", "spectrum_path", "spectrum_row"
]).drop_duplicates()

counts = verified.groupby("record_id").size()
accepted = verified.loc[
    verified["record_id"].isin(counts.index[counts.eq(1)])
]

for _, row in accepted.iterrows():
    rid = row["record_id"]
    mapped.at[rid, "spectrum_path"] = row["spectrum_path"]
    mapped.at[rid, "spectrum_row"] = row["spectrum_row"]
    mapped.at[rid, "mapping_note"] = (
        "Unique daily cumulative candidate at classified NIGHT; "
        "TARGETID verified in FIBERMAP and target NIGHT verified "
        "in EXP_FIBERMAP. Earlier exposures may also be included."
    )

mapped.to_csv(MAPPED_MANIFEST, index=False)

verified.to_csv(
    OUTPUT_ROOT / "prelim_verified_candidates.csv", index=False
)
pd.DataFrame(issues, columns=["record_id", "path", "reason"]).to_csv(
    OUTPUT_ROOT / "prelim_mapping_issues.csv", index=False
)

ready = (
    mapped["spectrum_path"].str.strip().ne("")
    & mapped["mapping_note"].str.strip().ne("")
)

print(f"Added {len(accepted)} preliminary mappings")
print(f"{ready.sum()} of {len(mapped)} records mapped")
print(f"{counts.gt(1).sum()} records have multiple verified candidates")
display(pd.DataFrame(issues).head(10))

Added 0 preliminary mappings
662 of 726 records mapped
0 records have multiple verified candidates


,record_id,path,reason
0,prelim_final_table:167,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
1,prelim_final_table:180,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
2,prelim_final_table:187,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
3,prelim_final_table:190,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
4,prelim_final_table:182,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
5,prelim_final_table:166,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
6,prelim_final_table:173,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
7,prelim_final_table:171,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
8,prelim_final_table:176,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...
9,prelim_final_table:179,/global/cfs/cdirs/desi/spectro/redux/daily/til...,extension not found: exp_fibermap (case insens...


In [8]:
import json
import numpy as np
import pandas as pd

mapped = pd.read_csv(
    MAPPED_MANIFEST, dtype=str, keep_default_na=False
)

loa_candidates = pd.read_csv(
    OUTPUT_ROOT / "onqueue_classifications_candidates.csv",
    dtype=str, keep_default_na=False,
)

unresolved_ids = set(
    mapped.loc[mapped["spectrum_path"].str.strip().eq(""), "record_id"]
)

loa_candidates = loa_candidates.loc[
    loa_candidates["record_id"].isin(unresolved_ids)
].copy()

loa_candidates["OBS_MJD"] = loa_candidates["provenance_json"].map(
    lambda text: float(json.loads(text)["OBS_MJD"])
)

for column in ["CAT_MIN_MJD", "CAT_MAX_MJD"]:
    loa_candidates[column] = pd.to_numeric(
        loa_candidates[column], errors="coerce"
    )

# Small tolerance accommodates rounding of OBS_MJD in the CSV.
tolerance = 1 / 86400  # one second
loa_candidates["time_within_coadd_range"] = (
    loa_candidates["OBS_MJD"].ge(
        loa_candidates["CAT_MIN_MJD"] - tolerance
    )
    & loa_candidates["OBS_MJD"].le(
        loa_candidates["CAT_MAX_MJD"] + tolerance
    )
)

print("Records with Loa candidates:",
      loa_candidates["record_id"].nunique())
print("Records with time inside a candidate's range:",
      loa_candidates.loc[
          loa_candidates["time_within_coadd_range"], "record_id"
      ].nunique())

display(loa_candidates[[
    "record_id", "TARGETID", "OBS_MJD",
    "CAT_MIN_MJD", "CAT_MAX_MJD", "CAT_COADD_NUMNIGHT",
    "time_within_coadd_range",
]].head(15))

Records with Loa candidates: 55
Records with time inside a candidate's range: 54


,record_id,TARGETID,OBS_MJD,CAT_MIN_MJD,CAT_MAX_MJD,CAT_COADD_NUMNIGHT,time_within_coadd_range
0,onqueue_classifications:2,2389938691637248,60373.30732,60373.307319,60373.307319,1,True
1,onqueue_classifications:8,2842536024473601,60195.89759,60195.408633,60196.386554,2,True
2,onqueue_classifications:48,39627651127706656,59387.36926,59387.361603,59387.376920,1,True
3,onqueue_classifications:49,39627652578940237,59609.08881,59609.088806,59609.088806,1,True
4,onqueue_classifications:51,39627663433799334,59377.96344,59377.469183,59378.457694,2,True
5,onqueue_classifications:54,39627703799780292,59378.18457,59378.184567,59378.184567,1,True
6,onqueue_classifications:63,39627757386207275,59655.60934,59651.281199,59664.250861,2,True
7,onqueue_classifications:71,39627838604709159,59490.18474,59490.184737,59490.184737,1,True
8,onqueue_classifications:73,39627848079639827,59619.38644,59619.386442,59619.386442,1,True
9,onqueue_classifications:77,39627862831009387,59871.16788,59871.167878,59871.167878,1,True


In [9]:
import json
import numpy as np
import pandas as pd
import fitsio

mapped = pd.read_csv(
    MAPPED_MANIFEST, dtype=str, keep_default_na=False
).set_index("record_id", drop=False)

checks = []

for path, group in loa_candidates.groupby("lookup_path"):
    try:
        fm = fitsio.read(
            path, ext="FIBERMAP",
            columns=["TARGETID", "TARGET_RA", "TARGET_DEC", "MEAN_MJD"],
        )

        for _, candidate in group.iterrows():
            rid = candidate["record_id"]
            if mapped.at[rid, "spectrum_path"].strip():
                continue

            result = {
                "record_id": rid,
                "spectrum_path": path,
                "spectrum_row": "",
                "separation_arcsec": np.nan,
                "time_difference_seconds": np.nan,
                "passes": False,
                "reason": "",
            }

            try:
                targetid = int(candidate["TARGETID"])
                indices = np.flatnonzero(fm["TARGETID"] == targetid)

                if len(indices) != 1:
                    raise ValueError(f"Found {len(indices)} TARGETID rows")

                index = int(indices[0])
                row = fm[index]
                original = json.loads(candidate["provenance_json"])
                obs_mjd = float(original["OBS_MJD"])

                # Great-circle position difference.
                ra1, dec1 = np.deg2rad([
                    float(candidate["RA"]), float(candidate["DEC"])
                ])
                ra2, dec2 = np.deg2rad([
                    float(row["TARGET_RA"]), float(row["TARGET_DEC"])
                ])
                a = (
                    np.sin((dec2 - dec1) / 2)**2
                    + np.cos(dec1) * np.cos(dec2)
                    * np.sin((ra2 - ra1) / 2)**2
                )
                separation = (
                    np.rad2deg(2 * np.arcsin(np.sqrt(np.clip(a, 0, 1))))
                    * 3600
                )
                time_difference = abs(
                    float(row["MEAN_MJD"]) - obs_mjd
                ) * 86400

                passes = (
                    np.isfinite(separation)
                    and np.isfinite(time_difference)
                    and separation <= 1.0
                    and time_difference <= 1.0
                )

                result.update(
                    spectrum_row=str(index),
                    separation_arcsec=separation,
                    time_difference_seconds=time_difference,
                    passes=passes,
                    reason="" if passes else "Position or mean-time mismatch",
                )

            except Exception as error:
                result["reason"] = str(error)

            checks.append(result)

    except Exception as error:
        checks.extend({
            "record_id": rid,
            "spectrum_path": path,
            "spectrum_row": "",
            "separation_arcsec": np.nan,
            "time_difference_seconds": np.nan,
            "passes": False,
            "reason": str(error),
        } for rid in group["record_id"])


loa_checks = pd.DataFrame(checks, columns=[
    "record_id", "spectrum_path", "spectrum_row",
    "separation_arcsec", "time_difference_seconds", "passes", "reason",
])

passing = loa_checks.loc[loa_checks["passes"].eq(True)].drop_duplicates(
    ["record_id", "spectrum_path", "spectrum_row"]
)
counts = passing.groupby("record_id").size()
accepted = passing.loc[
    passing["record_id"].isin(counts.index[counts.eq(1)])
]

for _, row in accepted.iterrows():
    rid = row["record_id"]
    mapped.at[rid, "spectrum_path"] = row["spectrum_path"]
    mapped.at[rid, "spectrum_row"] = row["spectrum_row"]
    mapped.at[rid, "mapping_note"] = (
        "Loa HEALPix coadd: exact TARGETID, position within 1 arcsec, "
        "and MEAN_MJD within 1 second of classified OBS_MJD; "
        "unique passing candidate."
    )

mapped.to_csv(MAPPED_MANIFEST, index=False)
loa_checks.to_csv(OUTPUT_ROOT / "onqueue_loa_checks.csv", index=False)

ready = (
    mapped["spectrum_path"].str.strip().ne("")
    & mapped["mapping_note"].str.strip().ne("")
)

print(f"Added {len(accepted)} Loa mappings")
print(f"{ready.sum()} of {len(mapped)} records mapped")
display(loa_checks.loc[~loa_checks["passes"].eq(True)])

Added 54 Loa mappings
716 of 726 records mapped


,record_id,spectrum_path,spectrum_row,separation_arcsec,time_difference_seconds,passes,reason
54,onqueue_classifications:152,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,116,0.000019,1.202311e+08,False,Position or mean-time mismatch
55,onqueue_classifications:205,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,135,0.000059,5.616662e+07,False,Position or mean-time mismatch


In [10]:
!rm -rf "{SPECTRA_DIR}"